## librerias

In [1]:
import os
import webbrowser
import pandas as pd
import json
import geopandas as gpd
import colorsys
import numpy as np
import http.server
import socketserver
from threading import Thread
import time
import hashlib
import unicodedata

## bases 

In [2]:
usuario = os.getlogin()

In [3]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

## back

In [4]:
base = base.drop(columns=['poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct'])


base.columns = (
    base.columns
        .str.strip()
        .str.lower()
        .map(
            lambda x: ''.join(
                c for c in unicodedata.normalize('NFD', x)
                if unicodedata.category(c) != 'Mn'
            )
        )
        .str.replace(" ", "_", regex=False)
)

In [5]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [6]:
base.columns

Index(['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada',
       'consultorios_generales', 'consultorios_generales_habilitados',
       'equipo_de_computo', 'banco_de_altura', 'banco_giratorio',
       'bascula_electronica__con_estadimetro', 'bascula_pesabebes_electronica',
       'bote_sanitario_con_pedal',
       'caja_portalaminilla_de_plastico_con_separadores',
       'carta_snellen_con_marco',
       'charola_de_mayo_de_acero_inoxidable,_dimensiones:_49x32cm',
       'cinta_metrica', 'contenedor_de_jabon_liquido',
       'contenedor_de_toallas_desechables',
       'contenedor_rigido_7.50_a_9.40_ml.',
       'cubeta_de_acero_inoxidable_y_bolsa', 'equipo_de_computo.1',
       'escritorio_medico_de_150x60x75', 'esfigmomanometro', 'espejo_vestidor',
       'espejos_graves_o_vaginales_chicos,_medianos_y_grandes',
       'estadimetro_pediatrico',
       'estetoscopio_capsula_doble._auxiliar_para_realizar_auscultacion',
       'estetoscopio_pinard_o_doppler_fetal_portati

In [7]:
# Obtener entidades únicas
entidades_unicas = sorted(base['entidad'].dropna().unique())

# Identificar columnas de equipamiento (todas excepto las que no son numéricas)
columnas_excluir = ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']
columnas_equipamiento = [col for col in base.columns if col not in columnas_excluir]

# Obtener unidades por entidad
def get_unidades_por_entidad(entidad):
    """Obtiene todas las unidades de una entidad con sus datos"""
    df_entidad = base[base['entidad'] == entidad]
    return df_entidad.to_dict('records')

In [8]:
# Configuración de colores
COLOR_PRIMARIO = "#FAF2F5"
COLOR_SECUNDARIO = '#AE8640'
COLOR_HBC = "#FDFDFDC0"
COLOR_FONDO = "#235B4E"
COLOR_BORDE = '#7A1737'
COLOR_TEXTO = '#000000'

In [9]:
# Función simple para formatear nombres de columnas
def formatear_nombre(columna):
    nombre = columna.replace('_', ' ')
    nombre = nombre.split('.')[0]
    palabras = nombre.split()
    palabras = [p.capitalize() for p in palabras]
    return ' '.join(palabras)


## front

In [10]:
script_url = "https://script.google.com/macros/s/AKfycbx0OqRe5XyRm4UL0BsU8WhErU_PwvnNfo0uZTKnyfHCmV4qGfImIn8hjPaAjSHIn_33/exec"

In [11]:
# Obtener columnas de equipamiento
columnas_equipamiento = [col for col in base.columns if col not in ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']]

html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Cuestionario de Equipamiento - IMSS Bienestar</title>
    <link href="https://fonts.googleapis.com/css2?family=League+Spartan:wght@400;500;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: "League Spartan", sans-serif;
        }}

        body {{
            background: linear-gradient(135deg, {COLOR_PRIMARIO} 0%, #fff 100%);
            min-height: 100vh;
            padding: 20px;
        }}

        .container {{
            width: 100%;
            max-width: 1400px;
            margin: 0 auto;
        }}

        .menu-container {{
            position: fixed;
            top: 20px;
            right: 20px;
            z-index: 1001;
        }}

        .menu-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            cursor: pointer;
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            gap: 6px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            transition: 0.3s;
        }}

        .menu-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: scale(1.05);
        }}

        .menu-btn span {{
            width: 25px;
            height: 3px;
            background: white;
            border-radius: 3px;
            transition: 0.3s;
        }}

        .menu-panel {{
            position: fixed;
            top: 0;
            right: -400px;
            width: 380px;
            height: 100%;
            background: white;
            box-shadow: -2px 0 10px rgba(0,0,0,0.1);
            z-index: 1002;
            transition: 0.3s;
            overflow-y: auto;
            padding: 80px 25px 25px 25px;
        }}

        .menu-panel.active {{
            right: 0;
        }}

        .menu-overlay {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            z-index: 1001;
            display: none;
        }}

        .menu-overlay.active {{
            display: block;
        }}

        .menu-panel h2 {{
            color: {COLOR_FONDO};
            margin-bottom: 20px;
            font-size: 24px;
            border-bottom: 3px solid {COLOR_SECUNDARIO};
            padding-bottom: 10px;
        }}

        .menu-panel h3 {{
            color: {COLOR_SECUNDARIO};
            margin: 20px 0 10px 0;
            font-size: 18px;
        }}

        .menu-panel p {{
            color: #333;
            line-height: 1.6;
            margin-bottom: 15px;
        }}

        .menu-panel ul, .menu-panel ol {{
            color: #555;
            margin-left: 20px;
            margin-bottom: 15px;
        }}

        .menu-panel li {{
            margin-bottom: 8px;
        }}

        .close-menu {{
            position: absolute;
            top: 20px;
            right: 20px;
            background: none;
            border: none;
            font-size: 30px;
            cursor: pointer;
            color: {COLOR_FONDO};
        }}

        .header {{
            background: {COLOR_FONDO};
            padding: 20px;
            color: white;
            margin-bottom: 30px;
            border-radius: 15px;
            text-align: center;
            position: relative;
        }}

        .header img {{
            height: 50px;
            margin-bottom: 10px;
        }}

        .header h1 {{
            font-size: 24px;
            margin-bottom: 5px;
        }}

        .instrucciones-rapidas {{
            background: white;
            border-radius: 12px;
            padding: 15px 20px;
            margin-bottom: 20px;
            display: flex;
            justify-content: center;
            gap: 30px;
            flex-wrap: wrap;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}

        .instruccion-item {{
            display: flex;
            align-items: center;
            gap: 12px;
            font-size: 14px;
            font-weight: 500;
        }}

        .instruccion-color {{
            width: 24px;
            height: 24px;
            border-radius: 6px;
        }}

        .color-verde {{
            background: #d4edda;
            border: 2px solid #2e7d32;
        }}

        .color-rojo {{
            background: #ffebee;
            border: 2px solid #c62828;
        }}

        .instruccion-texto {{
            color: #333;
        }}

        .instruccion-texto strong {{
            color: {COLOR_FONDO};
        }}

        .estados-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
            gap: 15px;
            max-height: 600px;
            overflow-y: auto;
            padding: 10px;
        }}

        .estado-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 12px;
            padding: 15px 10px;
            color: white;
            font-weight: 600;
            cursor: pointer;
            transition: 0.3s;
            font-size: 14px;
        }}

        .estado-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-3px);
        }}

        .modal {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.8);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal.active {{
            display: flex;
        }}
        
        .modal-form.active {{
            display: flex;
        }}

        .modal-content {{
            position: relative;
            background: linear-gradient(135deg, {COLOR_BORDE}dd);
            backdrop-filter: none;
            border-radius: 20px;
            padding: 30px;
            width: 90%;
            max-width: 450px;
            color: white;
            border: 1px solid rgba(255,255,255,0.2);
            animation: slideUp 0.3s;
        }}

        @keyframes slideUp {{
            from {{ transform: translateY(20px); opacity: 0; }}
            to {{ transform: translateY(0); opacity: 1; }}
        }}

        .modal h2 {{
            text-align: center;
            margin-bottom: 10px;
            font-size: 28px;
        }}

        .modal p {{
            text-align: center;
            margin-bottom: 20px;
            opacity: 0.9;
        }}

        .close-btn {{
            position: absolute;
            top: 10px;
            right: 15px;
            background: none;
            border: none;
            font-size: 28px;
            cursor: pointer;
            color: white;
            opacity: 0.8;
            transition: opacity 0.2s;
            line-height: 1;
        }}

        .close-btn:hover {{
            opacity: 1;
        }}

        .modal-form {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal-form.active {{
            display: flex;
        }}

        .modal-form-content {{
            position: relative;
            background: white;
            border-radius: 20px;
            padding: 30px;
            width: 95%;
            max-width: 1400px;
            max-height: 90vh;
            overflow-x: auto;
            overflow-y: auto;
        }}

        .modal-form-content .close-btn {{
            color: #333;
            top: 10px;
            right: 15px;
        }}

        .form-group {{
            margin-bottom: 15px;
        }}

        .form-group label {{
            display: block;
            margin-bottom: 5px;
            font-weight: 600;
        }}

        .form-group input, .form-group select {{
            width: 100%;
            padding: 10px;
            background: rgba(255,255,255,0.2);
            border: 1px solid rgba(255,255,255,0.3);
            border-radius: 8px;
            color: white;
            font-size: 14px;
        }}

        .form-group input::placeholder {{
            color: rgba(255,255,255,0.6);
        }}

        .form-group input:focus {{
            outline: none;
            border-color: white;
            background: rgba(255,255,255,0.25);
        }}

        .help-text {{
            font-size: 12px;
            margin-top: 5px;
            opacity: 0.8;
        }}

        .btn {{
            width: 100%;
            padding: 12px;
            background: white;
            color: {COLOR_FONDO};
            border: none;
            border-radius: 8px;
            font-weight: 700;
            font-size: 16px;
            cursor: pointer;
            transition: 0.3s;
            margin-top: 10px;
        }}

        .btn:hover {{
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.3);
        }}

        .error {{
            color: #ffcccc;
            font-size: 13px;
            margin-top: 10px;
            text-align: center;
            display: none;
        }}

        .table-container {{
            overflow-x: auto;
            overflow-y: auto;
            max-height: 60vh;
            position: relative;
        }}

        .unidades-table {{
            width: max-content;
            min-width: 100%;
            border-collapse: collapse;
            position: relative;
        }}

        .unidades-table th, .unidades-table td {{
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
            vertical-align: middle;
            white-space: nowrap;
        }}

        /* Columna fija 1: CLUES */
        .unidades-table th:nth-child(1),
        .unidades-table td:nth-child(1) {{
            position: sticky;
            left: 0;
            background-color: {COLOR_FONDO};
            z-index: 10;
            min-width: 100px;
        }}

        /* Columna fija 2: Unidad Medica */
        .unidades-table th:nth-child(2),
        .unidades-table td:nth-child(2) {{
            position: sticky;
            left: 100px;
            background-color: {COLOR_FONDO};
            z-index: 10;
            min-width: 250px;
        }}

        /* Encabezados fijos */
        .unidades-table th:nth-child(1),
        .unidades-table th:nth-child(2) {{
            background-color: {COLOR_FONDO};
            color: white;
            z-index: 20;
        }}

        /* Fondo de las celdas fijas */
        .unidades-table td:nth-child(1),
        .unidades-table td:nth-child(2) {{
            background-color: #f0f0f0;
            font-weight: bold;
        }}

        /* Sombra para la segunda columna fija */
        .unidades-table td:nth-child(2)::after,
        .unidades-table th:nth-child(2)::after {{
            content: '';
            position: absolute;
            top: 0;
            right: -5px;
            height: 100%;
            width: 5px;
            box-shadow: 2px 0 5px rgba(0,0,0,0.1);
            pointer-events: none;
        }}

        .unidades-table th {{
            background-color: {COLOR_FONDO};
            color: white;
            position: sticky;
            top: 0;
            transition: background-color 0.2s;
            z-index: 15;
        }}

        .unidades-table th:hover {{
            background-color: #1a8fbb !important;
            cursor: pointer;
        }}

        .unidades-table tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        .unidades-table tbody tr:hover {{
            background-color: #e3f2fd !important;
        }}

        .unidades-table tbody tr:hover td:first-child,
        .unidades-table tbody tr:hover td:nth-child(2) {{
            background-color: #bbdef5 !important;
        }}

        .unidades-table td:hover {{
            background-color: #fff3e0 !important;
        }}

        .unidades-table td:hover .valor-mostrado {{
            transform: scale(1.05);
            box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        }}

        /* Fila que ya fue guardada en la base de datos */
        .unidades-table tr.fila-guardada-bd {{
            background-color: #d4edda !important;
        }}

        .unidades-table tr.fila-guardada-bd:hover {{
            background-color: #c3e6cb !important;
        }}

        .unidades-table tr.fila-guardada-bd td:first-child,
        .unidades-table tr.fila-guardada-bd td:nth-child(2) {{
            background-color: #c3e6cb !important;
        }}

        .campo-container {{
            display: flex;
            gap: 8px;
            align-items: center;
            flex-wrap: wrap;
        }}

        .valor-mostrado {{
            display: inline-block;
            background: #e8f5e9;
            color: #2e7d32;
            padding: 5px 10px;
            border-radius: 5px;
            font-weight: bold;
            font-size: 14px;
            min-width: 50px;
            text-align: center;
            cursor: pointer;
            transition: 0.2s;
        }}

        .valor-mostrado:hover {{
            background: #c8e6c9;
            transform: scale(1.02);
        }}

        .valor-vacio {{
            background: #ffebee;
            color: #c62828;
            cursor: pointer;
        }}

        .valor-vacio:hover {{
            background: #ffcdd2;
        }}

        .input-edicion {{
            width: 80px;
            padding: 5px;
            border: 2px solid {COLOR_SECUNDARIO};
            border-radius: 4px;
            text-align: center;
            font-size: 14px;
        }}

        .btn-accion {{
            color: white;
            border: none;
            padding: 5px 10px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 12px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
            background: #2196F3;
        }}

        .btn-accion:hover {{
            background: #0b7dda;
        }}

        .btn-guardar-fila {{
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            padding: 6px 12px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 12px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
        }}

        .btn-guardar-fila:hover {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .btn-guardar-fila:disabled {{
            background: #999;
            cursor: not-allowed;
            transform: none;
        }}

        .btn-cancelar {{
            background: #666;
            color: white;
        }}

        .acciones {{
            display: flex;
            gap: 10px;
            margin-top: 20px;
            justify-content: center;
            flex-wrap: wrap;
        }}

        .progress {{
            margin-bottom: 20px;
            padding: 10px;
            background: #f0f0f0;
            border-radius: 8px;
            color: #333;
        }}

        .badge {{
            display: inline-block;
            padding: 3px 8px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: bold;
        }}
        
        .badge-success {{
            background: #4CAF50;
            color: white;
        }}
        
        .badge-warning {{
            background: #ff9800;
            color: white;
        }}

        .pagination {{
            display: flex;
            justify-content: center;
            align-items: center;
            gap: 10px;
            margin-top: 20px;
            margin-bottom: 20px;
        }}

        .pagination button {{
            background: {COLOR_FONDO};
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 8px;
            cursor: pointer;
            transition: 0.3s;
            font-weight: 600;
        }}

        .pagination button:hover:not(:disabled) {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-2px);
        }}

        .pagination button:disabled {{
            opacity: 0.5;
            cursor: not-allowed;
        }}

        .pagination span {{
            font-size: 14px;
            font-weight: 600;
        }}

        .page-info {{
            background: {COLOR_FONDO};
            color: white !important;
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 14px;
        }}

        .save-indicator {{
            position: fixed;
            bottom: 20px;
            right: 20px;
            background: #4CAF50;
            color: white;
            padding: 10px 15px;
            border-radius: 8px;
            font-size: 14px;
            opacity: 0;
            transition: opacity 0.3s;
            z-index: 1000;
        }}

        .equipo-tooltip {{
            position: fixed;
            background: #0b5a7c;
            color: white;
            padding: 8px 15px;
            border-radius: 8px;
            font-size: 14px;
            font-weight: bold;
            z-index: 2000;
            pointer-events: none;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            white-space: nowrap;
            font-family: "League Spartan", sans-serif;
        }}
    </style>
</head>
<body>
    <div class="menu-container">
        <button class="menu-btn" onclick="toggleMenu()">
            <span></span>
            <span></span>
            <span></span>
        </button>
    </div>

    <div class="menu-overlay" id="menuOverlay" onclick="toggleMenu()"></div>
    
    <div class="menu-panel" id="menuPanel">
        <button class="close-menu" onclick="toggleMenu()">&times;</button>
        <h2>Instrucciones</h2>
        
        <h3>1. Seleccionar Estado</h3>
        <p>Haga clic en el boton del estado correspondiente para comenzar el registro de equipamiento.</p>
        
        <h3>2. Registrar Datos del Usuario</h3>
        <p>Complete el formulario con nombre y correo electronico.</p>
        
        <h3>3. Llenar Equipamiento</h3>
        <p>Haga clic en cualquier celda para editarla, ingrese el valor y presione ACEPTAR.</p>
        <p><strong style="color:#2e7d32">VERDE:</strong> Valor ya registrado - Haga clic para modificar</p>
        <p><strong style="color:#c62828">ROJO:</strong> Campo pendiente - Haga clic para llenar</p>
        <p><strong style="color:#d4edda">FILA VERDE:</strong> Unidad ya guardada en la base de datos</p>
        
        <h3>4. Guardar Unidad</h3>
        <p>Una vez que haya llenado los campos de una unidad, presione el boton <strong>"Guardar unidad"</strong> al final de la fila.</p>
        
        <h3>5. Columnas Fijas</h3>
        <p>Las columnas CLUES y Unidad Medica permanecen fijas al desplazarse horizontalmente.</p>
        
        <h3>6. Progreso</h3>
        <p>La barra de progreso muestra el avance del llenado de todas las unidades.</p>
        
        <h3>Consejos</h3>
        <ul>
            <li>Use solo numeros enteros</li>
            <li>Debe presionar ACEPTAR en cada celda antes de guardar la fila</li>
            <li>El progreso local se guarda automaticamente</li>
            <li>Pase el mouse sobre cualquier celda para ver que equipo esta llenando</li>
            <li>Desplace horizontalmente para ver mas columnas de equipamiento</li>
        </ul>
        
        <h3>Soporte</h3>
        <p>Si tiene problemas, contacte al area de sistemas de IMSS Bienestar.</p>
    </div>

    <div class="container">
        <div class="header">
            <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" alt="IMSS Bienestar">
            <h1>CUESTIONARIO DE EQUIPAMIENTO POR UNIDAD MEDICA</h1>
            <p>Seleccione una entidad para registrar el equipamiento de sus unidades</p>
        </div>

        <div class="estados-grid" id="estadosGrid">
'''

# Generar botones de entidades
for entidad in entidades_unicas:
    num_unidades = len(base[base['entidad'] == entidad])
    html_content += f'''
            <button class="estado-btn" onclick="abrirModal('{entidad}')">
                {entidad}<br>
                <small style="font-size: 11px;">{num_unidades} unidades</small>
            </button>
    '''

html_content += f'''
        </div>
    </div>

    <div class="modal" id="loginModal">
        <div class="modal-content">
            <button class="close-btn" onclick="cerrarModal()">&times;</button>
            <h2 id="modalEstado"></h2>
            <p>Registre sus datos para continuar</p>
            
            <div class="form-group">
                <label>Entidad</label>
                <input type="text" id="usuarioInput" readonly>
            </div>
            
            <div class="form-group">
                <label>Nombre completo</label>
                <input type="text" id="nombreInput" placeholder="Ej: Juan Carlos Perez Gonzalez" required>
                <div class="help-text">Ingrese nombre(s) y apellidos completos</div>
            </div>
            
            <div class="form-group">
                <label>Correo electronico institucional</label>
                <input type="email" id="emailInput" placeholder="ejemplo@imssbienestar.gob.mx" required>
                <div class="help-text">Ingrese su correo electronico</div>
            </div>
            
            <button class="btn" onclick="validarDatos()">Continuar</button>
            <div id="errorMsg" class="error"></div>
        </div>
    </div>

    <div class="modal-form" id="formModal">
        <div class="modal-form-content">
            <button class="close-btn" onclick="cerrarFormModal()">&times;</button>
            <h2 id="formEstado"></h2>
            <div id="userInfo" style="background: #f0f0f0; padding: 10px; border-radius: 8px; margin-bottom: 15px; font-size: 14px;"></div>
            <div id="progressInfo" class="progress"></div>
            
            <div class="instrucciones-rapidas">
                <div class="instruccion-item">
                    <div class="instruccion-color color-verde"></div>
                    <div class="instruccion-texto"><strong>VERDE</strong> Haga clic para MODIFICAR</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-rojo"></div>
                    <div class="instruccion-texto"><strong>ROJO</strong> Haga clic para LLENAR</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #d4edda; width: 24px; height: 24px; border-radius: 6px; border: 1px solid #2e7d32;"></div>
                    <div class="instruccion-texto"><strong>FILA VERDE</strong> Unidad ya guardada</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #2196F3; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>ACEPTAR</strong> Para confirmar el valor</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: {COLOR_SECUNDARIO}; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>GUARDAR UNIDAD</strong> Envia los datos a la nube</div>
                </div>
            </div>
            
            <div class="table-container">
                <table class="unidades-table" id="unidadesTable">
                    <thead>
                        <tr>
                            <th style="min-width: 100px;">CLUES</th>
                            <th style="min-width: 250px;">Unidad Medica</th>
                            <th style="min-width: 150px;">Categoria</th>
                            {''.join([f'<th style="min-width: 180px;">{formatear_nombre(col)}</th>' for col in columnas_equipamiento])}
                            <th style="min-width: 120px;">Accion</th>
                        </tr>
                    </thead>
                    <tbody id="tableBody">
                    </tbody>
                </table>
            </div>
            
            <div class="pagination" id="pagination">
                <button onclick="cambiarPagina(-1)" id="btnAnterior">Anterior</button>
                <span id="paginaInfo"></span>
                <button onclick="cambiarPagina(1)" id="btnSiguiente">Siguiente</button>
            </div>
            
            <div class="acciones">
                <button type="button" class="btn btn-cancelar" onclick="cerrarFormModal()">Cerrar</button>
            </div>
        </div>
    </div>

    <div class="save-indicator" id="saveIndicator">
        Unidad guardada
    </div>

    <script>
        let celdaEnEdicion = null;
        let tooltip = null;
        let filasGuardadasBD = new Set();

        function crearTooltip() {{
            if (!tooltip) {{
                tooltip = document.createElement('div');
                tooltip.className = 'equipo-tooltip';
                tooltip.style.display = 'none';
                document.body.appendChild(tooltip);
            }}
            return tooltip;
        }}

        function mostrarTooltip(event, texto) {{
            const tooltip = crearTooltip();
            tooltip.textContent = texto;
            tooltip.style.display = 'block';
            let left = event.pageX + 15;
            let top = event.pageY - 30;
            
            if (left + tooltip.offsetWidth > window.innerWidth) {{
                left = event.pageX - tooltip.offsetWidth - 15;
            }}
            if (top < 0) {{
                top = event.pageY + 20;
            }}
            
            tooltip.style.left = left + 'px';
            tooltip.style.top = top + 'px';
        }}

        function ocultarTooltip() {{
            if (tooltip) {{
                tooltip.style.display = 'none';
            }}
        }}

        function toggleMenu() {{
            const panel = document.getElementById('menuPanel');
            const overlay = document.getElementById('menuOverlay');
            panel.classList.toggle('active');
            overlay.classList.toggle('active');
        }}

        document.addEventListener('keydown', function(e) {{
            if (e.key === 'Escape') {{
                const panel = document.getElementById('menuPanel');
                const overlay = document.getElementById('menuOverlay');
                panel.classList.remove('active');
                overlay.classList.remove('active');
                if (celdaEnEdicion === null) {{
                    cerrarModal();
                    cerrarFormModal();
                }}
                ocultarTooltip();
            }}
        }});

        const datosUnidades = {json.dumps({entidad: get_unidades_por_entidad(entidad) for entidad in entidades_unicas}, ensure_ascii=False)};
        const columnasEquipamiento = {json.dumps(columnas_equipamiento)};
        
        let estadoSeleccionado = '';
        let datosActuales = [];
        let usuarioActual = {{
            nombre: '',
            email: '',
            entidad: ''
        }};
        
        let paginaActual = 0;
        let unidadesPorPagina = 5;
        let totalPaginas = 0;

        function formatearNombreEquipo(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            if (formateado.length > 50) {{
                formateado = formateado.substring(0, 47) + '...';
            }}
            return formateado;
        }}

        function formatearNombre(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            return formateado;
        }}

        function guardarProgresoLocal() {{
            if (estadoSeleccionado && datosActuales.length > 0 && usuarioActual.email) {{
                const clave = `equipamiento_${{estadoSeleccionado}}_${{usuarioActual.email}}`;
                const progreso = {{
                    entidad: estadoSeleccionado,
                    usuario: usuarioActual,
                    datos: datosActuales,
                    fecha_guardado: new Date().toISOString()
                }};
                localStorage.setItem(clave, JSON.stringify(progreso));
            }}
        }}

        function cargarProgresoLocal(estado, email) {{
            const clave = `equipamiento_${{estado}}_${{email}}`;
            const guardado = localStorage.getItem(clave);
            if (guardado) {{
                try {{
                    const progreso = JSON.parse(guardado);
                    return progreso.datos;
                }} catch(e) {{
                    return null;
                }}
            }}
            return null;
        }}

        function verificarProgresoGuardado(estado, email) {{
            const clave = `equipamiento_${{estado}}_${{email}}`;
            const guardado = localStorage.getItem(clave);
            if (guardado) {{
                try {{
                    const progreso = JSON.parse(guardado);
                    const fecha = new Date(progreso.fecha_guardado);
                    return {{
                        existe: true,
                        fecha: fecha.toLocaleString()
                    }};
                }} catch(e) {{
                    return {{ existe: false }};
                }}
            }}
            return {{ existe: false }};
        }}

        function limpiarProgresoLocal() {{
            if (estadoSeleccionado && usuarioActual.email) {{
                const clave = `equipamiento_${{estadoSeleccionado}}_${{usuarioActual.email}}`;
                if (confirm('¿Esta seguro de que desea eliminar el progreso guardado?')) {{
                    localStorage.removeItem(clave);
                    alert('Progreso eliminado correctamente');
                    cargarUnidades(estadoSeleccionado);
                }}
            }}
        }}

        function abrirModal(estado) {{
            estadoSeleccionado = estado;
            document.getElementById('modalEstado').textContent = estado;
            document.getElementById('usuarioInput').value = estado;
            document.getElementById('nombreInput').value = '';
            document.getElementById('emailInput').value = '';
            document.getElementById('errorMsg').style.display = 'none';
            document.getElementById('loginModal').classList.add('active');
            document.getElementById('nombreInput').focus();
        }}

        function cerrarModal() {{
            document.getElementById('loginModal').classList.remove('active');
        }}

        function validarNombreCompleto(nombre) {{
            const nombreTrim = nombre.trim();
            const partes = nombreTrim.split(/\\s+/);
            if (partes.length < 2) return false;
            if (partes.length > 5) return false;
            for (let parte of partes) {{
                if (parte.length < 2) return false;
            }}
            return true;
        }}

        function validarDatos() {{
            const nombre = document.getElementById('nombreInput').value;
            const email = document.getElementById('emailInput').value.trim();
            
            if (!validarNombreCompleto(nombre)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese su nombre completo';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            if (email === '') {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            const emailRegex = /^[^\\s@]+@([^\\s@]+\\.)+[^\\s@]+$/;
            if (!emailRegex.test(email)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico valido';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            usuarioActual = {{
                nombre: nombre.trim(),
                email: email,
                entidad: estadoSeleccionado
            }};
            
            document.getElementById('errorMsg').style.display = 'none';
            cerrarModal();
            abrirFormulario(estadoSeleccionado);
        }}

        function abrirFormulario(estado) {{
            document.getElementById('formEstado').innerHTML = `Cuestionario de Equipamiento - <strong>${{estado}}</strong>`;
            document.getElementById('userInfo').innerHTML = `
                <strong>Registrado por:</strong> ${{usuarioActual.nombre}} | 
                <strong>Correo:</strong> ${{usuarioActual.email}} | 
                <strong>Entidad:</strong> ${{usuarioActual.entidad}}
            `;
            document.getElementById('formModal').classList.add('active');
            
            const progreso = verificarProgresoGuardado(estado, usuarioActual.email);
            
            if (progreso.existe) {{
                const restaurar = confirm(`Se encontro un progreso guardado del ${{progreso.fecha}}.\\n\\n¿Desea continuar donde lo dejo?`);
                if (restaurar) {{
                    cargarUnidadesConProgreso(estado);
                }} else {{
                    cargarUnidades(estado);
                }}
            }} else {{
                cargarUnidades(estado);
            }}
        }}

        function cargarUnidadesConProgreso(estado) {{
            const unidades = datosUnidades[estado] || [];
            const progreso = cargarProgresoLocal(estado, usuarioActual.email);
            
            if (progreso && progreso.length === unidades.length) {{
                datosActuales = JSON.parse(JSON.stringify(progreso));
            }} else {{
                datosActuales = JSON.parse(JSON.stringify(unidades));
            }}
            
            totalPaginas = Math.ceil(datosActuales.length / unidadesPorPagina);
            paginaActual = 0;
            
            if (totalPaginas > 0) {{
                mostrarPagina();
            }}
            actualizarProgreso();
            
            const progressInfo = document.getElementById('progressInfo');
            const mensajeRestaurado = document.createElement('div');
            mensajeRestaurado.style.background = '#e8f5e9';
            mensajeRestaurado.style.color = '#2e7d32';
            mensajeRestaurado.style.padding = '8px';
            mensajeRestaurado.style.borderRadius = '8px';
            mensajeRestaurado.style.marginTop = '10px';
            mensajeRestaurado.style.textAlign = 'center';
            mensajeRestaurado.innerHTML = 'Progreso anterior restaurado correctamente';
            progressInfo.appendChild(mensajeRestaurado);
            setTimeout(() => {{
                mensajeRestaurado.remove();
            }}, 3000);
        }}

        function guardarUnidadEnNube(filaIndex, btnElement) {{
            const unidad = datosActuales[filaIndex];
            const claveUnidad = usuarioActual.email + '|' + unidad.clues_imb;
            
            btnElement.disabled = true;
            btnElement.textContent = 'Guardando...';
            
            const datosParaGuardar = {{
                entidad: estadoSeleccionado,
                usuario_nombre: usuarioActual.nombre,
                usuario_email: usuarioActual.email,
                clues_imb: unidad.clues_imb,
                nombre_de_la_unidad: unidad.nombre_de_la_unidad,
                categoria: unidad.categoria_gerencial_ampliada
            }};
            
            for (let i = 0; i < columnasEquipamiento.length; i++) {{
                const col = columnasEquipamiento[i];
                const valor = unidad[col];
                datosParaGuardar[col] = (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') ? valor : '';
            }}
            
            const scriptURL = '{script_url}';
            
            fetch(scriptURL, {{ 
                method: 'POST', 
                mode: 'no-cors',
                headers: {{ 'Content-Type': 'application/json' }},
                body: JSON.stringify(datosParaGuardar)
            }})
            .then(() => {{
                filasGuardadasBD.add(claveUnidad);
                btnElement.disabled = false;
                btnElement.textContent = 'Guardado';
                btnElement.style.background = '#28a745';
                
                const row = btnElement.closest('tr');
                row.classList.add('fila-guardada-bd');
                
                const indicator = document.getElementById('saveIndicator');
                indicator.style.background = '#28a745';
                indicator.textContent = 'Unidad guardada en la nube';
                indicator.style.opacity = '1';
                setTimeout(() => {{
                    indicator.style.opacity = '0';
                    setTimeout(() => {{
                        btnElement.textContent = 'Guardar unidad';
                        btnElement.style.background = '{COLOR_SECUNDARIO}';
                    }}, 2000);
                }}, 2000);
            }})
            .catch(error => {{
                console.error('Error:', error);
                btnElement.disabled = false;
                btnElement.textContent = 'Guardar unidad';
                alert('Error al guardar. Revisa tu conexion.');
            }});
        }}

        function crearCampoEquipamiento(valorActual, filaOriginal, columna) {{
            const container = document.createElement('div');
            container.className = 'campo-container';
            
            const valorMostrado = document.createElement('div');
            valorMostrado.className = 'valor-mostrado';
            
            const tieneValor = (valorActual !== null && valorActual !== undefined && !isNaN(valorActual) && valorActual !== 0 && valorActual !== '');
            const valorDisplay = tieneValor ? valorActual : 'PENDIENTE';
            
            valorMostrado.textContent = valorDisplay;
            
            if (!tieneValor) {{
                valorMostrado.classList.add('valor-vacio');
            }}
            
            const iniciarEdicion = () => {{
                if (celdaEnEdicion !== null) {{
                    alert('Primero presione ACEPTAR en la celda que esta editando');
                    return;
                }}
                
                celdaEnEdicion = container;
                container.innerHTML = '';
                
                const inputField = document.createElement('input');
                inputField.type = 'number';
                inputField.step = '1';
                inputField.value = (tieneValor && valorActual !== null) ? valorActual : '';
                inputField.placeholder = '0';
                inputField.className = 'input-edicion';
                
                const btnAceptar = document.createElement('button');
                btnAceptar.className = 'btn-accion';
                btnAceptar.textContent = 'ACEPTAR';
                
                const finalizarEdicion = () => {{
                    const nuevoValor = inputField.value;
                    let numero = parseInt(nuevoValor);
                    
                    if (nuevoValor === '') {{
                        datosActuales[filaOriginal][columna] = null;
                    }} else if (!isNaN(numero) && numero >= 0) {{
                        datosActuales[filaOriginal][columna] = numero;
                    }} else {{
                        alert('Ingrese un numero valido');
                        return;
                    }}
                    
                    celdaEnEdicion = null;
                    actualizarProgreso();
                    mostrarPagina();
                    guardarProgresoLocal();
                }};
                
                btnAceptar.onclick = finalizarEdicion;
                inputField.onkeypress = (e) => {{
                    if (e.key === 'Enter') {{
                        finalizarEdicion();
                    }}
                }};
                
                container.appendChild(inputField);
                container.appendChild(btnAceptar);
                inputField.focus();
            }};
            
            valorMostrado.onclick = iniciarEdicion;
            container.appendChild(valorMostrado);
            
            return container;
        }}

        function cargarUnidades(estado) {{
            const unidades = datosUnidades[estado] || [];
            datosActuales = JSON.parse(JSON.stringify(unidades));
            filasGuardadasBD.clear();
            
            totalPaginas = Math.ceil(datosActuales.length / unidadesPorPagina);
            paginaActual = 0;
            
            if (totalPaginas > 0) {{
                mostrarPagina();
            }}
            actualizarProgreso();
        }}
        
        function mostrarPagina() {{
            const inicio = paginaActual * unidadesPorPagina;
            const fin = inicio + unidadesPorPagina;
            const unidadesPagina = datosActuales.slice(inicio, fin);
            
            const tbody = document.getElementById('tableBody');
            tbody.innerHTML = '';
            
            for (let idx = 0; idx < unidadesPagina.length; idx++) {{
                const unidad = unidadesPagina[idx];
                const filaOriginal = inicio + idx;
                const row = tbody.insertRow();
                
                const nombreUnidad = unidad.nombre_de_la_unidad || 'Sin nombre';
                const cluesUnidad = unidad.clues_imb || 'Sin CLUES';
                
                row.addEventListener('mouseenter', function(e) {{
                    mostrarTooltip(e, 'Unidad: ' + nombreUnidad + ' (' + cluesUnidad + ')');
                }});
                row.addEventListener('mousemove', function(e) {{
                    mostrarTooltip(e, 'Unidad: ' + nombreUnidad + ' (' + cluesUnidad + ')');
                }});
                row.addEventListener('mouseleave', function() {{
                    ocultarTooltip();
                }});
                
                const cellClues = row.insertCell(0);
                cellClues.innerHTML = '<strong>' + (unidad.clues_imb || '') + '</strong>';
                cellClues.style.backgroundColor = '#f0f0f0';
                
                const cellNombre = row.insertCell(1);
                cellNombre.innerHTML = unidad.nombre_de_la_unidad || '';
                cellNombre.style.backgroundColor = '#f0f0f0';
                
                const cellCategoria = row.insertCell(2);
                cellCategoria.innerHTML = unidad.categoria_gerencial_ampliada || '';
                cellCategoria.style.backgroundColor = '#f9f9f9';
                
                for (let i = 0; i < columnasEquipamiento.length; i++) {{
                    const col = columnasEquipamiento[i];
                    const cell = row.insertCell();
                    const valorActual = unidad[col];
                    const campo = crearCampoEquipamiento(valorActual, filaOriginal, col);
                    cell.appendChild(campo);
                    
                    const nombreEquipo = formatearNombreEquipo(col);
                    cell.addEventListener('mouseenter', function(e) {{
                        this.style.backgroundColor = '#fff3e0';
                        mostrarTooltip(e, nombreEquipo + '\\nUnidad: ' + nombreUnidad);
                    }});
                    cell.addEventListener('mousemove', function(e) {{
                        mostrarTooltip(e, nombreEquipo + '\\nUnidad: ' + nombreUnidad);
                    }});
                    cell.addEventListener('mouseleave', function() {{
                        this.style.backgroundColor = '';
                        ocultarTooltip();
                    }});
                }}
                
                const cellAccion = row.insertCell(columnasEquipamiento.length + 3);
                const btnGuardar = document.createElement('button');
                btnGuardar.className = 'btn-guardar-fila';
                btnGuardar.textContent = 'Guardar unidad';
                btnGuardar.onclick = (function(f, b) {{ 
                    return function() {{ guardarUnidadEnNube(f, b); }};
                }})(filaOriginal, btnGuardar);
                cellAccion.appendChild(btnGuardar);
                cellAccion.style.backgroundColor = '#f9f9f9';
                cellAccion.style.textAlign = 'center';
            }}
            
            const ths = document.querySelectorAll('.unidades-table th');
            for (let i = 0; i < ths.length; i++) {{
                const th = ths[i];
                if (i >= 3 && i < columnasEquipamiento.length + 3) {{
                    const nombreEquipo = th.textContent;
                    th.addEventListener('mouseenter', function(e) {{
                        this.style.backgroundColor = '#1a8fbb';
                        mostrarTooltip(e, 'Equipo: ' + nombreEquipo);
                    }});
                    th.addEventListener('mousemove', function(e) {{
                        mostrarTooltip(e, 'Equipo: ' + nombreEquipo);
                    }});
                    th.addEventListener('mouseleave', function() {{
                        this.style.backgroundColor = '';
                        ocultarTooltip();
                    }});
                }}
            }}
            
            const desde = inicio + 1;
            const hasta = Math.min(fin, datosActuales.length);
            document.getElementById('paginaInfo').innerHTML = '<span class="page-info">Pagina ' + (paginaActual + 1) + ' de ' + totalPaginas + ' | Mostrando ' + desde + ' - ' + hasta + ' de ' + datosActuales.length + ' unidades</span>';
            
            document.getElementById('btnAnterior').disabled = paginaActual === 0;
            document.getElementById('btnSiguiente').disabled = paginaActual === totalPaginas - 1;
        }}
        
        function cambiarPagina(direccion) {{
            if (celdaEnEdicion !== null) {{
                alert('Primero presione ACEPTAR en la celda que esta editando');
                return;
            }}
            const nuevaPagina = paginaActual + direccion;
            if (nuevaPagina >= 0 && nuevaPagina < totalPaginas) {{
                paginaActual = nuevaPagina;
                mostrarPagina();
            }}
        }}
        
        function actualizarProgreso() {{
            let completados = 0;
            let totalCampos = 0;
            
            for (let u = 0; u < datosActuales.length; u++) {{
                const unidad = datosActuales[u];
                for (let c = 0; c < columnasEquipamiento.length; c++) {{
                    const col = columnasEquipamiento[c];
                    totalCampos++;
                    const valor = unidad[col];
                    if (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') {{
                        completados++;
                    }}
                }}
            }}
            
            const porcentaje = totalCampos > 0 ? Math.round((completados / totalCampos) * 100) : 0;
            let badgeClass = porcentaje === 100 ? 'badge-success' : 'badge-warning';
            let badgeText = porcentaje === 100 ? 'COMPLETO' : 'PENDIENTE';
            
            document.getElementById('progressInfo').innerHTML = `
                <strong>Progreso de llenado:</strong>
                <div style="background: #ddd; border-radius: 10px; margin-top: 5px;">
                    <div style="background: {COLOR_SECUNDARIO}; width: ` + porcentaje + `%; height: 20px; border-radius: 10px; transition: width 0.3s;"></div>
                </div>
                <p style="margin-top: 5px;">` + completados + ` de ` + totalCampos + ` campos completados (` + porcentaje + `%)</p>
                <span class="badge ` + badgeClass + `">
                    ` + badgeText + `
                </span>
            `;
        }}

        function cerrarFormModal() {{
            if (celdaEnEdicion !== null) {{
                alert('Primero presione ACEPTAR en la celda que esta editando');
                return;
            }}
            document.getElementById('formModal').classList.remove('active');
            ocultarTooltip();
        }}

        document.getElementById('nombreInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});
        
        document.getElementById('emailInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});

        document.getElementById('loginModal').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarModal();
            }}
        }});

        document.getElementById('emailInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                validarDatos();
            }}
        }});
        
        document.getElementById('nombreInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                document.getElementById('emailInput').focus();
            }}
        }});
        
        document.addEventListener('mousemove', function(e) {{
            if (tooltip && tooltip.style.display === 'block') {{
                let left = e.pageX + 15;
                let top = e.pageY - 30;
                if (left + tooltip.offsetWidth > window.innerWidth) {{
                    left = e.pageX - tooltip.offsetWidth - 15;
                }}
                if (top < 0) {{
                    top = e.pageY + 20;
                }}
                tooltip.style.left = left + 'px';
                tooltip.style.top = top + 'px';
            }}
        }});
    </script>
</body>
</html>
'''

# Para guardar el archivo HTML
with open('cuestionario_equipamiento.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Archivo HTML generado: cuestionario_equipamiento.html")

Archivo HTML generado: cuestionario_equipamiento.html


## salida

In [12]:
# Guardar el archivo HTML
usuario = os.getlogin()
destino_html = fr"C:\Users\{usuario}\Downloads\formu\formulario_an\index.html"

try:
    with open(destino_html, "w", encoding="utf-8") as file:
        file.write(html_content)
    
   
    
    webbrowser.open(destino_html)
    
except Exception as e:
    
    destino_actual = os.path.join(os.getcwd(), "cuestionario_equipamiento_imss.html")
    with open(destino_actual, "w", encoding="utf-8") as file:
        file.write(html_content)
 
    webbrowser.open(destino_actual)